# Keystone Project — What Makes a Song Stream?
### Exploring the Most-Streamed Spotify Tracks of 2023

**Author:** Andy  
**Dataset:** Most Streamed Spotify Songs 2023 — Nidula Elgiriyewithana  
**Source:** https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023  
**DOI:** 10.34740/KAGGLE/DSV/6367938

---

## Project Question

> **What audio characteristics, release patterns, and platform factors are associated with high streaming volume among Spotify's most popular tracks?**

This notebook walks through the full data analysis process: loading raw data, inspecting it for quality issues, cleaning and restructuring it, exploring patterns through summary statistics and grouped analysis, and communicating key findings through polished visualizations.

The goal is not just to describe this dataset — it's to develop habits of thinking critically about what data can and cannot tell us, and to practice communicating findings clearly and honestly.

---


## Section 1: Environment Setup

All libraries are installed in a virtual environment. See `requirements.txt` in this repo for exact versions. The key libraries used here are `pandas` for data manipulation, `matplotlib` and `seaborn` for visualization, and `numpy` for numeric operations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Global chart style constants — applied consistently across all visuals
SOURCE_NOTE = "Source: Most Streamed Spotify Songs 2023 — N. Elgiriyewithana / Kaggle (DOI: 10.34740/KAGGLE/DSV/6367938)"
BLUE   = "#2563EB"
ORANGE = "#F97316"
GRAY   = "#6B7280"

def clean_axes(ax):
    """Remove top/right spines, add light horizontal grid. Applied to every chart."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.45)

def add_source(fig, extra_note=None):
    """Attach data source citation (and optional caveat) to bottom of figure."""
    fig.text(0.12, 0.01, SOURCE_NOTE, fontsize=7.5, color=GRAY)
    if extra_note:
        fig.text(0.12, -0.025, extra_note, fontsize=7.5, color="#B45309")

print("Libraries loaded.")


## Section 2: Loading the Data & First Look

The raw CSV is loaded first without any modifications so we can see exactly what we're working with before cleaning anything. This is an important habit — you want to understand the data's problems before you fix them, not discover them mid-analysis.

**About the encoding:** This file uses `latin-1` encoding rather than UTF-8. Some track names and artist names contain accented characters (e.g., Beyoncé, Björk) that will throw a `UnicodeDecodeError` with the default UTF-8 decoder. Specifying `encoding="latin-1"` handles this.


In [ ]:
# Load raw — no cleaning yet
df_raw = pd.read_csv("spotify-2023.csv", encoding="latin-1")

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
print("Column names:")
for col in df_raw.columns:
    print(f"  {col}")


In [ ]:
# First five rows — what does the raw data look like?
df_raw.head()


In [ ]:
# Data types and non-null counts for every column
df_raw.info()


### Initial observations

A few things stand out immediately from `.info()` and `.head()`:

1. **`streams` is an `object` (string) column**, not numeric. This means at least one value is non-numeric — it can't be cast automatically. We need to investigate and handle it.
2. **Several columns end in `_%`** (e.g., `danceability_%`, `energy_%`) — these are percentage values 0–100, already on a consistent scale, which is convenient.
3. **`key` has missing values** — not every track has a detected musical key. We'll need to decide whether to drop, fill, or work around those rows depending on what analysis uses `key`.
4. **`in_shazam_charts` also has missing values** — likely because not all tracks appear on Shazam at all.
5. **Release date is split across three columns** (`released_year`, `released_month`, `released_day`) rather than a single datetime — we'll combine these for any time-based analysis.


## Section 3: Data Cleaning & Wrangling

Good analysis requires trustworthy data. This section documents every cleaning decision and the reasoning behind it. The goal is a working copy (`df`) that is ready for EDA while preserving the raw original (`df_raw`) for reference.


### 3.1 — Investigate the `streams` Column

In [ ]:
# Which rows have non-numeric values in 'streams'?
non_numeric_streams = df_raw[pd.to_numeric(df_raw["streams"], errors="coerce").isna()]
print(f"Non-numeric 'streams' rows: {len(non_numeric_streams)}")
non_numeric_streams[["track_name", "artist(s)_name", "streams"]]


There is exactly one corrupted row. Its `streams` value is a concatenated string of other column values — it looks like a CSV formatting error where fields shifted. Since we cannot recover the true stream count for this track, and it's a single row out of ~953, we drop it.

This is a judgment call worth documenting: we're choosing to drop rather than impute because stream count is our primary outcome variable — guessing it would be worse than losing one row.


In [ ]:
# Working copy — start from raw
df = df_raw.copy()

# Cast streams to numeric; the one bad row becomes NaN, then we drop it
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")
df = df.dropna(subset=["streams"]).copy()

print(f"Rows after dropping corrupt stream value: {len(df):,}")


### 3.2 — Rename Columns for Readability

In [ ]:
# The artist column name has parentheses, which is annoying to type repeatedly
df = df.rename(columns={"artist(s)_name": "artist_name"})

# Confirm
print("Renamed columns:", [c for c in df.columns if "artist" in c])


### 3.3 — Build a Release Date Column

In [ ]:
# Combine year/month/day into a proper datetime for any time-series work
# errors="coerce" handles any edge cases (e.g., Feb 30) by producing NaT
df["release_date"] = pd.to_datetime(
    df[["released_year", "released_month", "released_day"]].rename(
        columns={"released_year": "year", "released_month": "month", "released_day": "day"}
    ),
    errors="coerce"
)

print(f"Release dates parsed: {df['release_date'].notna().sum():,} / {len(df):,}")
print(f"Date range: {df['release_date'].min().date()} → {df['release_date'].max().date()}")


### 3.4 — Check and Document Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing.to_string())
print()

# As a percentage of total rows
print("As % of dataset:")
print((missing / len(df) * 100).round(2).to_string())


**Decisions on missing values:**

- **`key`** (~2.5% missing): Musical key is used in one optional chart. We'll keep the NaN rows and filter them out only for that specific chart — dropping them entirely would remove valid data from all other analyses.
- **`in_shazam_charts`** (~5% missing): Likely means the track doesn't appear on Shazam charts at all, not that data is unavailable. We'll treat NaN as 0 for any analysis involving Shazam.
- **`release_date`** (if any NaT): Rows where the date couldn't be parsed are dropped only when we do time-series analysis.

We do **not** fill numeric feature columns (danceability, energy, etc.) with means or medians — there are no missing values there, and doing so without cause would be bad practice.


In [ ]:
# Fill Shazam NaN → 0 (absent from charts, not missing data)
df["in_shazam_charts"] = df["in_shazam_charts"].fillna(0).astype(int)

# Convenience columns used throughout the notebook
df["streams_B"] = df["streams"] / 1e9   # streams in billions (cleaner axis labels)

print("Cleaning complete.")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


### 3.5 — Add a Derived Grouping Column

In [ ]:
# Bin artist_count into three groups for comparison charts
df["artist_group"] = df["artist_count"].apply(
    lambda x: "Solo" if x == 1 else ("2 Artists" if x == 2 else "3+ Artists")
)

print(df["artist_group"].value_counts().to_string())


## Section 4: Ethics & Data Limitations

Before building any visuals, it's worth pausing to ask: *what can this data actually tell us, and what could it mislead us about?*

---

### What this dataset is — and isn't

This dataset contains the **most-streamed songs on Spotify in 2023**, not a random or representative sample of all music. Every single row is already a "winner." That matters for every conclusion we draw.

---

### Specific concerns

**Survivorship bias**  
We cannot use this data to study what makes songs *fail* to gain traction, because unsuccessful songs simply aren't here. Saying "danceable songs get more streams" based on this data is like surveying lottery winners to learn how to get rich — the sample is pre-filtered by the outcome we're trying to explain.

**Playlist placement ≠ organic popularity**  
`in_spotify_playlists` is strongly correlated with stream counts — but playlist curation is not neutral. Major-label artists have documented advantages in securing Spotify editorial playlist placements. A chart showing "more playlists = more streams" could look like a story about audio quality when it's partly a story about industry relationships and algorithmic amplification.

**Streams ≠ unique listeners**  
One superfan streaming a track 1,000 times counts identically to 1,000 unique listeners. High stream counts for certain artists may reflect intense fanbases (e.g., Taylor Swift's Swifties, K-pop fandoms) as much as broad general reach.

**Geographic representation gaps**  
Even "global" Spotify charts skew toward English-language and Latin markets. Artists from South Asia, Sub-Saharan Africa, the Middle East, and much of East Asia are largely invisible here — not because their music isn't popular locally, but because Spotify's user distribution is uneven globally. Any claim about "what the world listens to" based on this data would overstate.

**Recency effects**  
Older tracks in this dataset have had more calendar time to accumulate streams. A song from 2017 has had 6+ years to build up; a 2023 release has had months. Raw stream counts are not directly comparable across release years.

---

### Bottom line for all charts that follow

> *Findings apply to this sample of already-successful tracks — not to music broadly, not to all listeners globally. Correlation patterns here describe what top-streamed hits have in common, not what causes streaming success.*


## Section 5: Exploratory Data Analysis

### 5.1 — Summary Statistics

Starting with `.describe()` gives a quick statistical profile of every numeric column. This is the first real look at ranges, medians, and potential outliers.


In [ ]:
# Summary stats for key numeric columns
cols_of_interest = [
    "streams_B", "bpm", "danceability_%", "energy_%", "valence_%",
    "acousticness_%", "speechiness_%", "liveness_%",
    "in_spotify_playlists", "in_apple_playlists", "in_spotify_charts"
]
df[cols_of_interest].describe().round(2)


A few things stand out:

- **`streams_B`**: Mean is much higher than the median, confirming a right-skewed distribution — a small number of mega-hits dramatically inflate the average. The median is a more honest "typical" value here.
- **`in_spotify_playlists`**: Max is in the thousands. Some tracks appear in an enormous number of playlists — likely major editorial playlists that themselves have massive reach.
- **`valence_%`**: Spread is wide (std ~25), meaning the dataset includes a full range from very melancholy to very upbeat tracks. This isn't a "only happy songs go viral" situation.
- **`speechiness_%`**: Very low mean (~10), which makes sense — most top tracks are sung, not spoken.


### 5.2 — Grouped Analysis: Stream Counts by Release Year

In [ ]:
yearly_streams = (
    df.groupby("released_year")["streams_B"]
    .agg(["median", "mean", "count"])
    .rename(columns={"median": "Median Streams (B)", "mean": "Mean Streams (B)", "count": "Track Count"})
    .sort_index()
)

# Show years with at least 5 tracks
print(yearly_streams[yearly_streams["Track Count"] >= 5].round(3).to_string())


The median stream count is highest for older tracks (2017–2019 range) — not because those years produced better music, but because those tracks have had more time to accumulate streams. This is the **recency effect** mentioned in the ethics section, and it's visible right here in the numbers.

This is exactly why raw stream counts should not be used to compare songs across release years without context.


### 5.3 — Grouped Analysis: Audio Features by Mode (Major vs. Minor)

In [ ]:
mode_stats = (
    df.groupby("mode")[["danceability_%", "energy_%", "valence_%", "acousticness_%"]]
    .agg(["median", "count"])
)
print(mode_stats.round(2).to_string())


Minor-key tracks have a noticeably lower median valence (positivity/happiness) than major-key tracks — which matches music theory intuition. But the difference in danceability and energy is much smaller, suggesting that "danceable" and "energetic" don't require a happy tone.

This is worth visualizing more clearly — see Story Chart 3.


### 5.4 — Top 10 Artists by Total Streams

In [ ]:
top_artists = (
    df.groupby("artist_name")["streams_B"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_artists.columns = ["Artist", "Total Streams (B)"]
top_artists["Total Streams (B)"] = top_artists["Total Streams (B)"].round(2)
print(top_artists.to_string(index=False))


**Important caveat on this table:** These are *total* streams from tracks that appear in the 2023 most-streamed list. An artist with one massive long-running hit (e.g., The Weeknd's "Blinding Lights") may rank above an artist who released many moderately successful tracks this year. This measures catalog dominance as much as 2023 activity.


## Section 6: EDA Visualizations

These three charts are exploratory — their purpose is to orient us to the data's shape and surface patterns worth investigating further. Each follows the Week 4 design principles: y-axes starting at zero, neutral purposeful color, descriptive neutral titles, and labeled axes with units.


### EDA Chart 1 — Distribution of Stream Counts

**What I'm trying to see:** Is the distribution of stream counts roughly symmetric, or are there a small number of massive outliers pulling the mean up?

**Why a histogram:** Histograms show the shape of a continuous distribution — the thing a single mean or median number hides.

**Design choices:** Bins are sized to show meaningful variation without hiding the right tail. y-axis starts at zero. Single neutral blue — no rainbow coloring that would imply categories that don't exist here.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df["streams_B"], bins=45, color=BLUE, edgecolor="white", linewidth=0.4)

# Mark median and mean with vertical lines to make the skew visible
median_val = df["streams_B"].median()
mean_val   = df["streams_B"].mean()
ax.axvline(median_val, color="#16A34A", linewidth=1.8, linestyle="--", label=f"Median: {median_val:.2f}B")
ax.axvline(mean_val,   color=ORANGE,   linewidth=1.8, linestyle=":",  label=f"Mean: {mean_val:.2f}B")

ax.set_xlabel("Streams (billions)", fontsize=11)
ax.set_ylabel("Number of Tracks", fontsize=11)
ax.set_title("Distribution of Stream Counts — Top Spotify Tracks, 2023", fontsize=13, pad=12)
ax.legend(fontsize=9, frameon=False)
clean_axes(ax)
add_source(fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


### EDA Chart 2 — How Many Top-Streamed Tracks Per Release Year?

**What I'm trying to see:** Which release years contribute the most tracks to Spotify's 2023 most-streamed list?

**Why a bar chart:** One category (year) vs. one count — bar charts are the clearest choice. Years are sorted chronologically, which is the natural order for time data.

**Design choices:** Value labels on top of each bar so exact counts are readable without hovering. Bars are a uniform blue — color variation here would imply something meaningful about the years that isn't there.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

year_counts = df["released_year"].value_counts().sort_index()
bars = ax.bar(year_counts.index.astype(str), year_counts.values,
              color=BLUE, width=0.62, edgecolor="white")

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1.5, str(int(h)),
            ha="center", va="bottom", fontsize=8, color="#374151")

ax.set_xlabel("Release Year", fontsize=11)
ax.set_ylabel("Number of Tracks in 2023 Most-Streamed List", fontsize=11)
ax.set_title("Tracks in 2023 Most-Streamed List by Release Year", fontsize=13, pad=12)
ax.set_ylim(0, year_counts.max() * 1.18)
clean_axes(ax)
add_source(fig, "Note: Older tracks have had more time to accumulate streams, which influences their presence here.")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


### EDA Chart 3 — Audio Feature Profiles Across All Tracks (Box Plots)

**What I'm trying to see:** What is the typical range and spread of each audio feature? Are any features tightly clustered (most top songs are similar) or widely spread (top songs vary a lot)?

**Why box plots:** Six features side by side — a box plot per feature shows median, IQR, and outliers in a compact, comparable format. Six separate histograms would be harder to compare.

**Design choices:** All features share the same 0–100 scale (they're already percentages), so placing them on one axis is valid and direct. Uniform color — the categories here are the features themselves, so we let position, not color, encode the grouping.


In [ ]:
feature_cols = ["danceability_%", "energy_%", "valence_%", "acousticness_%", "speechiness_%", "liveness_%"]
feature_labels = ["Danceability", "Energy", "Valence", "Acousticness", "Speechiness", "Liveness"]

plot_data = df[feature_cols].copy()
plot_data.columns = feature_labels

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [plot_data[col].dropna() for col in feature_labels],
    labels=feature_labels,
    patch_artist=True,
    medianprops=dict(color="white", linewidth=2.2),
    whiskerprops=dict(color=GRAY, linewidth=1),
    capprops=dict(color=GRAY, linewidth=1),
    flierprops=dict(marker="o", markersize=3, alpha=0.3, markerfacecolor=GRAY, markeredgewidth=0)
)
for patch in bp["boxes"]:
    patch.set_facecolor(BLUE)
    patch.set_alpha(0.72)

ax.set_ylabel("Feature Value (0–100 scale)", fontsize=11)
ax.set_title("Audio Feature Distributions — Top Spotify Tracks, 2023", fontsize=13, pad=12)
clean_axes(ax)
add_source(fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


**What I learned:** Danceability, energy, and valence all show wide spreads — top-streamed songs are not all the same "type." Speechiness is consistently low (most tracks are sung, not spoken). Acousticness is low-median with a wide spread, meaning acoustic tracks do appear in the top-streamed list but less commonly. There's no single audio fingerprint for a hit song.


## Section 7: Polished Story Visualizations

These charts are intentional. Each one is built around a specific question, designed to communicate clearly to someone who hasn't seen the data, and accompanied by an explanation of the design choices and what the chart reveals.


### Story Chart 1 — Does Playlist Presence Predict Streaming Volume?

**Question:** Is there a relationship between how many Spotify playlists a track appears in and how many streams it accumulates?

**Why this chart type:** A scatter plot shows the relationship between two continuous variables — the thing a bar chart or table can't show. Adding a trend line highlights the direction of the relationship without overstating the strength of it.

**Design choices:**
- Semi-transparent points (alpha=0.25) reveal density — where many tracks overlap, the cluster darkens naturally
- Log scale on the y-axis (streams) because the distribution is heavily right-skewed; a linear scale would compress 95% of points into a narrow band
- Trend line in orange to stand out from the blue scatter without using an aggressive color
- A printed caution note below the chart acknowledges the correlation-causation limitation directly

**Accuracy:** y-axis minimum is set by the data (log scale), not truncated. The trend line is linear on log-scale, which is appropriate for this type of relationship. The caution note is part of the chart — not just a footnote the reader might miss.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = df["in_spotify_playlists"].dropna()
y = df.loc[x.index, "streams"]
mask = (x > 0) & (y > 0)
x, y = x[mask], y[mask]
log_y = np.log10(y)

ax.scatter(x, log_y, alpha=0.22, s=16, color=BLUE, linewidths=0)

# Trend line (linear fit on log10 streams)
z = np.polyfit(x, log_y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.quantile(0.97), 300)
ax.plot(x_line, p(x_line), color=ORANGE, linewidth=2.2, label="Linear trend")

ax.set_xlabel("Number of Spotify Playlists Track Appears In", fontsize=11)
ax.set_ylabel("Streams (log₁₀ scale)", fontsize=11)
ax.set_title("Playlist Presence vs. Streaming Volume — Top Spotify Tracks, 2023", fontsize=13, pad=12)

ytick_vals = [7, 7.5, 8, 8.5, 9, 9.5, 10]
ax.set_yticks(ytick_vals)
ax.set_yticklabels([f"10^{v}" for v in ytick_vals])

ax.legend(fontsize=9, frameon=False)
clean_axes(ax)
ax.grid(axis="both", linestyle="--", linewidth=0.45, alpha=0.35)

add_source(fig, "⚠ Correlation ≠ causation. Playlist placement may drive streams, popular tracks may earn more placements — or both.")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


**What this tells us:** There is a positive relationship — tracks that appear in more playlists tend to accumulate more streams. But the scatter is wide, meaning many tracks appear in few playlists and still stream heavily, and playlist count alone doesn't determine streaming success. The caution on causality is real: Spotify's editorial team may add already-popular tracks to more playlists, creating a feedback loop rather than a clean cause-effect relationship.


### Story Chart 2 — Solo Tracks vs. Collaborations: Stream Count Comparison

**Question:** Do tracks with multiple credited artists tend to accumulate more streams than solo tracks?

**Why this chart type:** Box plots by group make medians and spreads directly comparable. A bar chart showing only group means would hide the wide variance within each group — which is the most interesting part of the story here.

**Design choices:**
- Three groups (Solo / 2 Artists / 3+ Artists), ordered naturally
- Sample size printed in the x-tick label so the reader knows how much data each box represents
- y-axis capped at the 99th percentile to prevent a few extreme outliers from compressing all the boxes into the bottom of the chart, while outliers are still plotted as individual points
- Streams in billions for readable axis labels

**Accuracy:** y-axis starts at zero. No truncation. Outliers are shown, not hidden.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

order = ["Solo", "2 Artists", "3+ Artists"]
group_data = [df.loc[df["artist_group"] == g, "streams_B"].dropna() for g in order]
counts = [len(g) for g in group_data]

bp = ax.boxplot(
    group_data,
    labels=[f"{g}\n(n={c})" for g, c in zip(order, counts)],
    patch_artist=True,
    medianprops=dict(color="white", linewidth=2.5),
    whiskerprops=dict(color=GRAY, linewidth=1),
    capprops=dict(color=GRAY, linewidth=1),
    flierprops=dict(marker="o", markersize=4, alpha=0.3,
                    markerfacecolor=GRAY, markeredgewidth=0)
)
for patch in bp["boxes"]:
    patch.set_facecolor(BLUE)
    patch.set_alpha(0.72)

ax.set_ylim(0, df["streams_B"].quantile(0.99) * 1.18)
ax.set_ylabel("Streams (billions)", fontsize=11)
ax.set_title("Stream Counts by Number of Credited Artists — Top Spotify Tracks, 2023", fontsize=13, pad=12)
clean_axes(ax)
add_source(fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


**What this tells us:** The medians are similar across groups — collaborations don't dramatically outperform solo tracks at the median. However, the spread is notably wider for 2-artist tracks, suggesting that successful collabs can hit higher ceilings while less successful ones perform similarly to solo tracks. The 3+ artist group is small (n is much lower), so conclusions there are less reliable.


### Story Chart 3 — Valence (Happiness) by Musical Mode: Major vs. Minor

**Question:** Do songs in a major key actually sound "happier" (higher valence) than songs in a minor key, at least among top-streamed tracks?

**Why this chart type:** Overlapping density curves (KDE plots) show the full shape of each group's distribution — where most values fall, whether there's a single peak or multiple, and how much the groups overlap. A bar chart showing only means would miss all of this.

**Design choices:**
- Two overlapping curves, one per mode, with fill and slight transparency so both are visible where they overlap
- Colors chosen for contrast and accessibility (blue vs. orange, not red vs. green)
- Vertical dashed lines mark each group's median for direct comparison
- x-axis labeled with plain English ("Low valence → High valence") rather than just "valence_%"

**Accuracy:** No truncation. The full 0–100 range is shown. Medians are accurately computed and labeled.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

major = df.loc[df["mode"] == "Major", "valence_%"].dropna()
minor = df.loc[df["mode"] == "Minor", "valence_%"].dropna()

major.plot.kde(ax=ax, color=BLUE,   label=f"Major key  (n={len(major):,})", linewidth=2.2)
minor.plot.kde(ax=ax, color=ORANGE, label=f"Minor key  (n={len(minor):,})", linewidth=2.2)

# Shade under each curve
from matplotlib.patches import Patch
x_range = np.linspace(0, 100, 500)
from scipy.stats import gaussian_kde
for data, color in [(major, BLUE), (minor, ORANGE)]:
    kde = gaussian_kde(data)
    ax.fill_between(x_range, kde(x_range), alpha=0.08, color=color)

# Median lines
ax.axvline(major.median(), color=BLUE,   linewidth=1.5, linestyle="--",
           label=f"Major median: {major.median():.0f}")
ax.axvline(minor.median(), color=ORANGE, linewidth=1.5, linestyle="--",
           label=f"Minor median: {minor.median():.0f}")

ax.set_xlim(0, 100)
ax.set_xlabel("Valence (0 = Low / Sad  →  100 = High / Happy)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Song 'Happiness' (Valence) by Musical Mode — Top Spotify Tracks, 2023", fontsize=13, pad=12)
ax.legend(fontsize=9, frameon=False)
clean_axes(ax)
add_source(fig)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


**What this tells us:** Major-key tracks do skew toward higher valence (more positive/happy tone), and minor-key tracks peak lower — which matches music theory. But the overlap is enormous. Plenty of major-key songs have low valence, and plenty of minor-key songs sound upbeat. Mode alone is a weak predictor of emotional tone. This is a good example of where a simple "major = happy, minor = sad" narrative would oversimplify what the data actually shows.


## Section 8: Summary & What's Next

### What I found

- **Stream counts are highly skewed** — a small number of mega-hits account for a disproportionate share of streams. The median is a much more honest "typical" value than the mean.
- **Playlist presence correlates with streaming volume**, but the relationship has wide variance and direction of causality is unclear.
- **Collaborations don't dramatically outperform solo tracks** at the median — but 2-artist tracks show a wider spread, suggesting the ceiling is higher.
- **Major-key tracks have higher median valence**, confirming music theory intuition — but the overlap between modes is large enough that mode alone predicts very little about a song's emotional tone.
- **There is no single audio fingerprint for a hit song.** Danceability, energy, and valence all vary widely among top-streamed tracks.

### What the data can't tell us

- Why these songs became popular — the dataset captures outcomes, not causes
- Anything about songs that *didn't* make the most-streamed list (survivorship bias)
- Whether findings generalize globally, given Spotify's uneven geographic reach

### Next steps (toward the full capstone)

- **Merge with Billboard Hot 100** — which tracks dominate Spotify but not radio, and vice versa?
- **Release month seasonality** — do certain months produce more long-running hits?
- **Deeper artist-level analysis** — audio feature "fingerprints" for top artists
- **Cross-platform correlation** — does Apple playlist count or Deezer chart presence independently predict Spotify streams?
